In [1]:
import os
import glob

import json
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
import numpy as np 
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon,box

In [2]:
FILEPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training'

EXCELPATH = r'C:\Users\b1114674\Desktop\thesis\AI4EO_data\international-charter-eo4ai-etq-challenge\Training\Phase_1_Dataset_Details.xlsx'

DESTINATION_PRE_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\PRE'
DESTINATION_POST_PATH = r'C:\Users\b1114674\Desktop\thesis\data\TRAINING_DATA\POST'

In [ ]:
class GEO_PreProssessing:

    def __init__(self):
          pass
        

    def open(self,path,no_channels=[1,2,3],transpose=True):
        '''
        Read the image 

        Args:
        path(str): filepath to the image
        no_channels(list(int)): how many channels do we need
        transpose(bool): True to change the shape of image to (H,W,C)

        Return: Image in numpy array format as (H,W,C) and Source format
        '''
        src = rasterio.open(path)
        img = src.read(no_channels)
        if transpose:
              img.transpose(1,2,0)
              return img,src
        else:
              return img,src

    def findOverlap(self,in_pre,in_post,out_pre,out_post,no_channels=[1,2,3]):
        
        '''
        Find the overlapping area an save both raster within intersection extent

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        
        '''

        _,pre_src = self.open(in_pre,no_channels)
        pre_bounds = box(*pre_src.bounds)
        crs = pre_src.crs
        
        _,post_src = self.open(in_post,no_channels)
        post_bounds = box(*post_src.bounds)

        common_area = pre_bounds.intersection(post_bounds)
    

        assert not common_area.is_empty, f'No Overlapping areas in the given images {os.path.basename(in_pre)} and {os.path.basename(in_post)}'

        # Converts Shapely geometry to GeoJSON format
        common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

        pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
        pre_meta = pre_src.meta.copy()
        pre_meta.update({
            "height": pre_clip.shape[1],
            "width": pre_clip.shape[2],
            "count": pre_clip.shape[0],
            "transform": pre_transform
            })
    

        post_clip, post_transform = mask(post_src, common_geojson, crop=True)
        post_meta = post_src.meta.copy()
        post_meta.update({
            "height": post_clip.shape[1],
            "width": post_clip.shape[2],
            "count": post_clip.shape[0],
            "transform": post_transform
            })
        
        try:
            with rasterio.open(out_pre,'w',**pre_meta) as pre_dest:
                                pre_dest.write(pre_clip)

            with rasterio.open(out_post,'w',**post_meta) as post_dest:
                                post_dest.write(post_clip)

        except ValueError:
              print('Invalid filepath!')


    def makeChips(self,in_pre,in_post,out_pre,out_post,no_channels=[1,2,3],chip_size=224,stride=224):    

        '''

        Args:
        in_Pre(str) : Absolute filepath to the pre image file
        in_post(str) : Absolute filepath to the post image file
        out_pre(str) : Absolute filepath to the Save cropped pre image file
        out_post(str) : Absolute filepath to the Save cropped post image file
        no_channels(list(int)): how many channels do we need
        chip_size(int): The size of the image height and width
        stride(int): step size or overlap between the chips
        
        '''
        with rasterio.open(in_pre) as pre_src, rasterio.open(in_post) as post_src:
            counter = 0

            if pre_src.height == post_src.height and pre_src.width == post_src.width:

                for row in range(0,pre_src.height,stride):
                        for col in range(0,pre_src.width,stride):
                            window = Window(col,row,chip_size,chip_size)

                            pre_img = pre_src.read(no_channels,window=window)
                            post_img = post_src.read(no_channels,window=window)

                            if row + chip_size > pre_src.height or col + chip_size > pre_src.width:
                                continue

                            pre_meta = pre_src.meta.copy()
                            pre_meta.update({
                                "height":pre_img.shape[1],
                                "width":pre_img.shape[2],
                                "count":pre_img.shape[0],
                                "transform":rasterio.windows.transform(window,pre_src.transform)  
                            })

                            post_meta = post_src.meta.copy()
                            post_meta.update({
                                "height":post_img.shape[1],
                                "width":post_img.shape[2],
                                "count":post_img.shape[0],
                                "transform":rasterio.windows.transform(window,post_src.transform)  
                            })
                            
                            try:

                                if os.path.exists(out_pre) and os.path.exists(out_post):

                                    with rasterio.open(os.path.join(out_pre,f"pre{counter}.tif"),"w",**pre_meta) as pre_dest:
                                        pre_dest.write(pre_img)

                                    with rasterio.open(os.path.join(out_post,f"post{counter}.tif"),"w",**post_meta) as post_dest:
                                        post_dest.write(post_img)

                                    counter+=1

                            except ValueError:
                                print("Invalid path!")
            else:
                print('[OVERLAP ERROR]----- MISMATCH IN IMAGE SIZES')

                crs = pre_src.crs
                pre_meta = pre_src.meta.copy()
                post_meta = post_src.meta.copy()
                pre_bound = box(*pre_src.bounds)
                post_bound = box(*post_src.bounds)

                common_area = pre_bound.intersection(post_bound)

                common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

                print(f'Before PRE crop {pre_src.height,pre_src.width,pre_src.count}')

                print(f'Before POST crop {post_src.height,post_src.width,post_src.count}')


                pre_clip,pre_transform = mask(pre_src,common_geojson,crop=True)
                post_clip,post_transform = mask(post_src,common_geojson,crop=True)

                print('IMAGE CROPPED TO A COMMON SIZE')

                assert pre_clip.shape[1] == post_clip.shape[1] and pre_clip.shape[2] == post_clip.shape[2], f"ISSUE WITH THE IMAGE SIZING, TRY TO CUT THE IMAGE INTO A COMMAN SIZE {pre_clip.shape} and {post_clip.shape}"

                for i in range(0,pre_clip.shape[1],stride):
                        for j in range(0,pre_clip.shape[2],stride):
                            pre_chip = pre_clip[:len(no_channels),i:i+chip_size,j:j+chip_size]
                            post_chip= post_clip[:len(no_channels),i:i+chip_size,j:j+chip_size]

                            pre_meta.update({
                                    "height":pre_chip.shape[1],
                                "width":pre_chip.shape[2],
                                "count":pre_chip.shape[0],
                                "transform": pre_transform
                                    
                            })

                            post_meta.update({
                                    "height":post_chip.shape[1],
                                "width":post_chip.shape[2],
                                "count":post_chip.shape[0],
                                "transform": post_transform
                                    
                            })

                            try:

                                if os.path.exists(out_pre) and os.path.exists(out_post):

                                    with rasterio.open(os.path.join(out_pre,f"pre{counter}.tif"),"w",**pre_meta) as pre_dest:
                                        pre_dest.write(pre_chip)

                                    with rasterio.open(os.path.join(out_post,f"post{counter}.tif"),"w",**post_meta) as post_dest:
                                        post_dest.write(post_chip)

                                    counter+=1

                            except ValueError:
                                print("Invalid path!")
                                
                                
    def Mean_Std(self,IMAGE_FOLDER_PATH,no_channels):
        '''
        Computing the Mean and Std for the whole dataset

        Args:
            IMAGE_FOLDER_PATH(str): Provide the directory consist of images (SAMPLE: ./ROOT_DATA_FOLDER/PRE_FOLDER)
        '''
        
        try:

            for path in os.listdir(IMAGE_FOLDER_PATH):
                img,_= open(path,no_channels,transpose=False)
                pixels = img.reshape(img.shape[0], -1).T 

                mean = np.zeros(pixels.shape[1])
                M2 = np.zeros(pixels.shape[1])

                n = pixels.shape[0]
                new_mean = np.mean(pixels, axis=0)
                new_M2 = np.var(pixels, axis=0) * n

                delta = new_mean - mean
                total_n = n_pixels + n
                mean += delta * n / total_n
                M2 += new_M2 + delta**2 * n_pixels * n / total_n
                n_pixels = total_n

            std = np.sqrt(M2 / n_pixels)
            print(f'[MEAN]----{mean}')
            print(f'[STD]----{std}')
        
        except ValueError:
            print('PATH is invalid !')

In [30]:
process = GEO_PreProssessing()

In [31]:
folderpath =r'C:\Users\b1114674\Desktop\thesis\data\AI4EO'

In [32]:
prelist = [
    os.path.join(folderpath, 'pre', img)
    for img in os.listdir(os.path.join(folderpath, 'pre'))
    if img.endswith('.tif')
]

postlist = [
    os.path.join(folderpath, 'post', img)
    for img in os.listdir(os.path.join(folderpath, 'post'))
    if img.endswith('.tif')
]

In [33]:
for i in range(0,len(prelist)):
    process.makeChips(prelist[i],postlist[i],r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippre', r'C:\Users\b1114674\Desktop\thesis\data\AI4EO\chippost')



[OVERLAP ERROR]----- MISMATCH IN IMAGE SIZES
Before PRE crop (6515, 4945, 4)
Before POST crop (6512, 4930, 4)


c:\Users\b1114674\AppData\Local\miniconda3\envs\play\lib\site-packages\geopandas\array.py:340: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  self.crs = crs
c:\Users\b1114674\AppData\Local\miniconda3\envs\play\lib\site-packages\geopandas\geoseries.py:244: UserWarning: Cannot set the CRS, falling back to None. The CRS support requires the 'pyproj' package, but it is not installed or does not import correctly. The functions depending on CRS will raise an error or may produce unexpected results.
  self.geometry.values.crs = value


IMAGE CROPPED TO A COMMON SIZE


AssertionError: ISSUE WITH THE IMAGE SIZING, TRY TO CUT THE IMAGE INTO A COMMAN SIZE (4, 6515, 4945) and (4, 6512, 4930)

In [ ]:
import os
import json
import rasterio
from rasterio.mask import mask
from rasterio.windows import Window
from rasterio.warp import reproject, calculate_default_transform, Resampling
from shapely.geometry import box
import geopandas as gpd
import numpy as np


class GEO_PreProcessing:

    def __init__(self):
        pass

    # ------------------ ALIGN FUNCTION (no saving) ------------------
    def align_to_ref(self, ref_src, src_src):
        """
        Align src raster to ref raster grid (CRS, resolution, extent).
        Returns aligned array and its updated transform.
        """
        transform, width, height = calculate_default_transform(
            src_src.crs, ref_src.crs, ref_src.width, ref_src.height, *ref_src.bounds
        )

        aligned_data = np.empty((src_src.count, ref_src.height, ref_src.width), dtype=src_src.dtypes[0])

        for i in range(src_src.count):
            reproject(
                source=rasterio.band(src_src, i + 1),
                destination=aligned_data[i],
                src_transform=src_src.transform,
                src_crs=src_src.crs,
                dst_transform=ref_src.transform,
                dst_crs=ref_src.crs,
                resampling=Resampling.bilinear
            )

        print("[ALIGN DONE] Aligned image to reference grid (in-memory).")
        return aligned_data, ref_src.transform

    # ------------------ MAKE CHIP FUNCTION ------------------
    def makeChips(self, in_pre, in_post, out_pre, out_post, no_channels=[1, 2, 3], chip_size=224, stride=224):
        """
        Creates chips from pre and post images, automatically aligning or cropping if needed.
        Returns chips that are spatially aligned, without saving aligned intermediates.
        """
        os.makedirs(out_pre, exist_ok=True)
        os.makedirs(out_post, exist_ok=True)

        with rasterio.open(in_pre) as pre_src, rasterio.open(in_post) as post_src:
            counter = 0

            # ---------------- CASE 1: SAME SIZE ----------------
            if pre_src.height == post_src.height and pre_src.width == post_src.width:
                print("[INFO] Images are already same size. Proceeding with chipping...")
                for row in range(0, pre_src.height - chip_size + 1, stride):
                    for col in range(0, pre_src.width - chip_size + 1, stride):
                        window = Window(col, row, chip_size, chip_size)
                        pre_img = pre_src.read(no_channels, window=window)
                        post_img = post_src.read(no_channels, window=window)

                        pre_meta = pre_src.meta.copy()
                        pre_meta.update({
                            "height": chip_size,
                            "width": chip_size,
                            "count": len(no_channels),
                            "transform": rasterio.windows.transform(window, pre_src.transform)
                        })
                        post_meta = pre_meta.copy()
                        post_meta["transform"] = rasterio.windows.transform(window, post_src.transform)

                        with rasterio.open(os.path.join(out_pre, f"pre_{counter}.tif"), "w", **pre_meta) as pre_dest:
                            pre_dest.write(pre_img)
                        with rasterio.open(os.path.join(out_post, f"post_{counter}.tif"), "w", **post_meta) as post_dest:
                            post_dest.write(post_img)
                        counter += 1

            # ---------------- CASE 2: DIFFERENT SIZE ----------------
            else:
                print("[MISMATCH] Image sizes differ. Finding intersection...")

                pre_bound = box(*pre_src.bounds)
                post_bound = box(*post_src.bounds)
                common_area = pre_bound.intersection(post_bound)

                if not common_area.is_empty:
                    print("[INFO] Found common intersection. Cropping both rasters...")
                    crs = pre_src.crs
                    common_geojson = [json.loads(gpd.GeoSeries([common_area], crs=crs).to_json())['features'][0]['geometry']]

                    pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
                    post_clip, post_transform = mask(post_src, common_geojson, crop=True)

                    # If still not same, align post to pre
                    if pre_clip.shape[1] != post_clip.shape[1] or pre_clip.shape[2] != post_clip.shape[2]:
                        print("[WARNING] Cropped rasters mismatched. Aligning in-memory...")
                        aligned_post, aligned_transform = self.align_to_ref(pre_src, post_src)
                        post_clip, _ = mask(
                            rasterio.io.MemoryFile().open(
                                driver="GTiff",
                                height=aligned_post.shape[1],
                                width=aligned_post.shape[2],
                                count=aligned_post.shape[0],
                                dtype=aligned_post.dtype.name,
                                crs=pre_src.crs,
                                transform=pre_src.transform
                            ),
                            common_geojson,
                            crop=True
                        )
                        post_transform = pre_transform

                    print(f"[INFO] Cropped to common size: {pre_clip.shape}")

                    # Chip both
                    for i in range(0, pre_clip.shape[1] - chip_size + 1, stride):
                        for j in range(0, pre_clip.shape[2] - chip_size + 1, stride):
                            pre_chip = pre_clip[:len(no_channels), i:i+chip_size, j:j+chip_size]
                            post_chip = post_clip[:len(no_channels), i:i+chip_size, j:j+chip_size]

                            if pre_chip.shape[1] != chip_size or pre_chip.shape[2] != chip_size:
                                continue

                            pre_meta = pre_src.meta.copy()
                            pre_meta.update({
                                "height": chip_size,
                                "width": chip_size,
                                "count": len(no_channels),
                                "transform": pre_transform
                            })
                            post_meta = pre_meta.copy()
                            post_meta["transform"] = post_transform

                            with rasterio.open(os.path.join(out_pre, f"pre_{counter}.tif"), "w", **pre_meta) as pre_dest:
                                pre_dest.write(pre_chip)
                            with rasterio.open(os.path.join(out_post, f"post_{counter}.tif"), "w", **post_meta) as post_dest:
                                post_dest.write(post_chip)
                            counter += 1

                else:
                    print("[ERROR] No intersection found. Aligning and retrying...")
                    aligned_post, _ = self.align_to_ref(pre_src, post_src)
                    # Re-mask using pre bounds (as intersection)
                    common_geojson = [json.loads(gpd.GeoSeries([pre_bound], crs=pre_src.crs).to_json())['features'][0]['geometry']]

                    pre_clip, pre_transform = mask(pre_src, common_geojson, crop=True)
                    post_clip, _ = mask(
                        rasterio.io.MemoryFile().open(
                            driver="GTiff",
                            height=aligned_post.shape[1],
                            width=aligned_post.shape[2],
                            count=aligned_post.shape[0],
                            dtype=aligned_post.dtype.name,
                            crs=pre_src.crs,
                            transform=pre_src.transform
                        ),
                        common_geojson,
                        crop=True
                    )

                    print(f"[INFO] Retry successful. Cropped to: {pre_clip.shape}")

                    # Chip again
                    for i in range(0, pre_clip.shape[1] - chip_size + 1, stride):
                        for j in range(0, pre_clip.shape[2] - chip_size + 1, stride):
                            pre_chip = pre_clip[:len(no_channels), i:i+chip_size, j:j+chip_size]
                            post_chip = post_clip[:len(no_channels), i:i+chip_size, j:j+chip_size]

                            if pre_chip.shape[1] != chip_size or pre_chip.shape[2] != chip_size:
                                continue

                            pre_meta = pre_src.meta.copy()
                            pre_meta.update({
                                "height": chip_size,
                                "width": chip_size,
                                "count": len(no_channels),
                                "transform": pre_transform
                            })
                            post_meta = pre_meta.copy()
                            post_meta["transform"] = pre_transform

                            with rasterio.open(os.path.join(out_pre, f"pre_{counter}.tif"), "w", **pre_meta) as pre_dest:
                                pre_dest.write(pre_chip)
                            with rasterio.open(os.path.join(out_post, f"post_{counter}.tif"), "w", **post_meta) as post_dest:
                                post_dest.write(post_chip)
                            counter += 1

            print(f"[DONE] Created {counter} aligned chip pairs.")
